# Retriever Fine-tuning: 의료 질문으로 Dense·Reranker 학습

`LabeledBaseline.ipynb`의 **Labeled + DB 보강** 시스템에서 검색 모델 두 개를 의료 질문으로 파인튜닝하고, 같은 시험 문제로 전후를 비교합니다.

| 모델 | 원래 모델 | 학습 방식 |
|---|---|---|
| Dense | `BAAI/bge-base-en-v1.5` | (질문, 정답 passage, 헷갈리는 오답 passage) 묶음으로 질문과 정답이 가까워지게 학습 |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-6-v2` | (질문, passage) 쌍이 정답인지 아닌지 맞히도록 학습 |

**학습 데이터**: train 질문 중 라벨이 A/B이고 정답 passage가 보강 DB에 있는 질문입니다. 헷갈리는 오답(hard negative)은 원래 Dense 모델이 상위에 올렸지만 정답이 아닌 passage에서 고릅니다.

**시험 문제가 학습에 섞이지 않게 하는 규칙**
- 학습은 train 질문만 씁니다. 평가는 LabeledBaseline과 같은 validation 세트(`lab_med_test`, `lab_med_ext_test`, `lab_ood_test`)로 합니다.
- 임계값 튜닝 세트(`lab_med_tune`, `lab_med_ext_tune`)는 학습에서 뺍니다. 학습한 질문으로 임계값을 정하면 점수가 부풀기 때문입니다.
- validation 질문과 문장이 같은 train 질문은 뺍니다.
- validation 정답 passage는 오답 후보로 쓰지 않습니다.

**사전 조건**: `QueryClassification.ipynb`를 train `med_url` 전체(`N_TRAIN_MED = None`)로 실행해 `data/labels/query_labels.parquet`가 있어야 합니다. 런타임은 A100 또는 L4입니다.

## 0. 환경 설정

In [ ]:
%pip install -q -U datasets bm25s PyStemmer faiss-cpu sentence-transformers accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re, json, random, hashlib
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ===== 경로 =====
BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", "/content/drive/MyDrive/RAG-Chat"))
DATA_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "index"
MODEL_DIR = BASE_DIR / "models"
RESULT_DIR = BASE_DIR / "results" / "finetune"
LABEL_PATH = BASE_DIR / "data" / "labels" / "query_labels.parquet"
for d in (DATA_DIR, INDEX_DIR, MODEL_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== 데이터와 검색 설정 (LabeledBaseline과 동일) =====
SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"
N_OOD_TEST = 1000
N_TUNE = 500
EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
K_CAND = 100
K_RERANK = 50
K_EVAL = 10
RRF_K = 60

# ===== 파인튜닝 설정 =====
NEG_DEPTH = 50         # 원래 Dense 검색 상위 몇 개에서 오답 후보를 고를지
NEG_SKIP = 3           # 최상위 몇 개는 건너뜀 (정답 표시가 없을 뿐 실제로는 정답인 passage일 수 있어서)
N_NEG = 4              # 질문당 오답 수 (Dense는 첫 번째만, Reranker는 전부 사용)
EMBED_EPOCHS, EMBED_BATCH, EMBED_LR = 1, 64, 2e-5
RERANK_EPOCHS, RERANK_BATCH, RERANK_LR = 1, 32, 2e-5
EMBED_FT_DIR = MODEL_DIR / "bge-base-en-v1.5-med-ft"
RERANK_FT_DIR = MODEL_DIR / "ms-marco-MiniLM-L-6-v2-med-ft"

random.seed(SEED)
np.random.seed(SEED)

## 1. 데이터 준비 (LabeledBaseline과 동일 코드)
코퍼스, 평가 세트, 보강 DB를 LabeledBaseline과 똑같이 만듭니다. 같은 라벨 파일을 쓰면 시험 문제도 같습니다.

In [ ]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

In [ ]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

In [ ]:
MEDICAL_KEYWORDS = [
    "disease", "disorder", "syndrome", "diabetes", "cancer", "asthma", "pneumonia", "infection",
    "arthritis", "stroke", "hypertension", "anemia", "symptom", "pain", "fever", "headache",
    "nausea", "vomiting", "dizziness", "fatigue", "swelling", "bleeding", "diagnosis", "diagnose",
    "blood test", "biopsy", "mri", "ct scan", "x-ray", "ultrasound", "medication", "medicine",
    "drug", "antibiotic", "insulin", "vaccine", "treatment", "therapy", "surgery", "prescription",
    "side effect", "heart", "brain", "kidney", "liver", "lung", "blood pressure", "hormone",
    "immune system", "pregnancy", "depression", "anxiety", "adhd", "mental health", "bipolar",
]
keyword_pattern = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k) for k in MEDICAL_KEYWORDS) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def build_query_sets(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    gold = (
        fs[fs["is_medical"] & (fs["is_selected"] == 1)]
        .groupby("query_id")["text"]
        .apply(lambda ts: sorted({text2pid[t] for t in ts}))
    )
    any_med = set(fs.loc[fs["is_medical"], "query_id"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    answerable = df["answers"].map(has_answer)

    med = base[answerable & base["query_id"].isin(gold.index)].copy()
    med["gold_pids"] = med["query_id"].map(gold)

    ood = base[
        answerable
        & ~base["query_id"].isin(any_med)
        & ~base["query"].str.contains(keyword_pattern)
    ].copy()
    ood["gold_pids"] = [[] for _ in range(len(ood))]
    return med.reset_index(drop=True), ood.reset_index(drop=True)

med_val, ood_val = build_query_sets("validation")
med_train, ood_train = build_query_sets("train")

qsets = {
    "med_test": med_val,
    "ood_test": ood_val.sample(N_OOD_TEST, random_state=SEED).reset_index(drop=True),
    "med_tune": med_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
    "ood_tune": ood_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
}
for name, q in qsets.items():
    print(f"{name:9s} {len(q):5d}")
display(qsets["med_test"].head())

In [ ]:
labels = pd.read_parquet(LABEL_PATH)
url_sets = qsets   # 1-1에서 만든 URL 기준 세트
FLAG_COLS = ["label", "high_risk", "time_sensitive"]

def med_with_label(med_df, split, label):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "med_url") & (labels["label"] == label)]
    return med_df.merge(lab[["query_id"] + FLAG_COLS], on="query_id").reset_index(drop=True)

def ood_with_label(split, n):
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "C")]
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = [[] for _ in range(len(out))]
    return out.sample(min(n, len(out)), random_state=SEED).reset_index(drop=True)

lab_med_train = med_with_label(med_train, "train", "A")
qsets = {
    "url_med_test": url_sets["med_test"],
    "url_med_tune": url_sets["med_tune"],
    "url_ood_tune": url_sets["ood_tune"],
    "lab_med_test": med_with_label(med_val, "validation", "A"),
    "lab_b_test": med_with_label(med_val, "validation", "B"),
    "lab_ood_test": ood_with_label("validation", N_OOD_TEST),
    "lab_med_tune": lab_med_train.sample(min(N_TUNE, len(lab_med_train)), random_state=SEED).reset_index(drop=True),
    "lab_ood_tune": ood_with_label("train", N_TUNE),
}
for name, q in qsets.items():
    print(f"{name:13s} {len(q):5d}")
print("\nvalidation med_url 라벨 분포:")
print(labels[(labels["split"] == "validation") & (labels["pool"] == "med_url")]["label"].value_counts().to_string())

In [ ]:
# 시험(validation) 질문의 라벨은 DB에 반영하지 않음: DB는 질문보다 먼저 만들어지는 것이므로
med_keys = labels.loc[(labels["split"] == "train") & labels["label"].isin(["A", "B"]), ["split", "query_id"]]
in_med_q = pd.MultiIndex.from_frame(flat[["split", "query_id"]]).isin(list(med_keys.itertuples(index=False, name=None)))
extra = (
    flat[in_med_q & ~flat["text"].isin(text2pid)]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
extra.insert(0, "pid", np.arange(len(corpus), len(corpus) + len(extra)))
aug_corpus = pd.concat([corpus, extra], ignore_index=True)   # 행 번호 == pid
aug_text2pid = dict(zip(aug_corpus["text"], aug_corpus["pid"]))

def ext_med_set(split, need_gold):
    """정답이 일반 사이트에만 있는 A 질문. 정답 passage가 보강 DB에 있으면 그 pid를 표시"""
    lab = labels[(labels["split"] == split) & (labels["pool"] == "no_med") & (labels["label"] == "A")]
    fs = flat[(flat["split"] == split) & (flat["is_selected"] == 1)]
    gold = fs.groupby("query_id")["text"].apply(lambda ts: sorted({aug_text2pid[t] for t in ts if t in aug_text2pid}))
    out = lab[["query_id", "query", "query_type", "answers"] + FLAG_COLS].copy()
    out["answers"] = out["answers"].map(list)
    out["gold_pids"] = out["query_id"].map(gold).map(lambda g: g if isinstance(g, list) else [])
    if need_gold:   # 임계값 튜닝용 train 질문은 정답 passage가 DB에 있어야 함
        out = out[out["gold_pids"].map(len) > 0]
    return out.reset_index(drop=True)

qsets["lab_med_ext_test"] = ext_med_set("validation", need_gold=False)
ext_tune = ext_med_set("train", need_gold=True)
qsets["lab_med_ext_tune"] = ext_tune.sample(min(N_TUNE, len(ext_tune)), random_state=SEED).reset_index(drop=True)

print(f"URL DB {len(corpus):,} → 보강 DB {len(aug_corpus):,} (+{len(extra):,})")
print(extra["host"].value_counts().head(10).to_string())
for name in ["lab_med_ext_test", "lab_med_ext_tune"]:
    print(f"{name:17s} {len(qsets[name]):5d}")
in_db = qsets["lab_med_ext_test"]["gold_pids"].map(len) > 0
print(f"lab_med_ext_test 중 정답 passage가 보강 DB에 있는 질문: {in_db.sum()} / {len(in_db)}")

## 2. 검색 도구 (보강 DB)
LabeledBaseline의 검색 코드와 같고, Dense 모델과 Reranker를 바꿔 끼울 수 있게 인자로 받습니다.

In [ ]:
import bm25s, Stemmer

stemmer = Stemmer.Stemmer("english")
N_URL = len(corpus)
passages = aug_corpus["text"].tolist()   # 보강 DB (앞 N_URL개 = URL DB)

bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(passages, stopwords="en", stemmer=stemmer))

def bm25_search(queries, k=K_CAND):
    tokens = bm25s.tokenize(queries, stopwords="en", stemmer=stemmer, show_progress=False)
    ids, scores = bm25.retrieve(tokens, k=k, show_progress=False)
    return ids, scores

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

def encode_passages(embedder, texts, path=None):
    if path is not None and path.exists():
        return np.load(path)
    emb = embedder.encode(
        texts, batch_size=256, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    ).astype("float32")
    if path is not None:
        np.save(path, emb)
    return emb

def build_index(emb):
    index = faiss.IndexFlatIP(emb.shape[1])   # 정규화된 벡터의 내적 = 코사인 유사도
    index.add(emb)
    return index

def dense_search(embedder, index, queries, k=K_CAND):
    q_emb = embedder.encode(
        [QUERY_PREFIX + q for q in queries], batch_size=256,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    scores, ids = index.search(q_emb, k)
    return ids, scores

# 원래 모델: LabeledBaseline과 같은 임베딩 캐시를 씀
base_embedder = SentenceTransformer(EMBED_MODEL, device="cuda").half()
model_tag = EMBED_MODEL.split("/")[-1]
extra_tag = hashlib.md5("\n".join(passages[N_URL:]).encode()).hexdigest()[:8]
base_index = build_index(np.concatenate([
    encode_passages(base_embedder, passages[:N_URL], INDEX_DIR / f"{model_tag}_{N_URL}.npy"),
    encode_passages(base_embedder, passages[N_URL:], INDEX_DIR / f"{model_tag}_aug_{extra_tag}.npy"),
]))
base_reranker = CrossEncoder(RERANK_MODEL, device="cuda", max_length=512)

In [ ]:
def rrf_fuse(*id_lists_per_run, k=RRF_K, top=K_RERANK):
    """run마다 (n_query, K) 형태의 id 배열을 받아 질의별로 RRF로 합친 상위 top개 pid 리스트를 반환."""
    fused = []
    for per_query in zip(*id_lists_per_run):
        score = defaultdict(float)
        for ids in per_query:
            for rank, pid in enumerate(ids):
                score[int(pid)] += 1.0 / (k + rank + 1)
        fused.append(sorted(score, key=score.get, reverse=True)[:top])
    return fused

In [ ]:
def rerank(reranker, queries, candidates):
    """candidates: 질의별 pid 리스트 → (정렬된 pid 리스트, 정렬된 점수 리스트)"""
    pairs = [(q, passages[pid]) for q, cand in zip(queries, candidates) for pid in cand]
    flat_scores = reranker.predict(pairs, batch_size=256, show_progress_bar=False)
    out_ids, out_scores, pos = [], [], 0
    for cand in candidates:
        s = np.asarray(flat_scores[pos:pos + len(cand)])
        pos += len(cand)
        order = np.argsort(-s)
        out_ids.append([cand[i] for i in order])
        out_scores.append(s[order].tolist())
    return out_ids, out_scores

## 3. 파인튜닝 데이터
- **질문**: train에서 라벨이 A/B이고 정답 passage가 보강 DB에 있는 질문 (임계값 튜닝 세트와 validation과 같은 문장의 질문은 제외)
- **정답**: 그 질문의 `is_selected=1` passage
- **오답**: 원래 Dense 모델의 검색 결과 `NEG_SKIP`~`NEG_DEPTH`위 중에서, 같은 질문에 딸린 passage와 validation 정답 passage를 뺀 것. 같은 질문에 딸린 passage는 정답 표시가 없어도 관련 있는 경우가 많아서 오답으로 쓰지 않습니다.

In [ ]:
tune_ids = set(qsets["lab_med_tune"]["query_id"]) | set(qsets["lab_med_ext_tune"]["query_id"])
val_queries = set(qa["validation"]["query"].map(normalize_text).str.lower())

ft_tr = flat[flat["split"] == "train"]
gold = (ft_tr[ft_tr["is_selected"] == 1].groupby("query_id")["text"]
        .apply(lambda ts: sorted({aug_text2pid[t] for t in ts if t in aug_text2pid})))
own = ft_tr.groupby("query_id")["text"].apply(lambda ts: {aug_text2pid[t] for t in ts if t in aug_text2pid})
fv = flat[(flat["split"] == "validation") & (flat["is_selected"] == 1)]
val_gold = {aug_text2pid[t] for t in fv["text"] if t in aug_text2pid}

lab_tr = labels[(labels["split"] == "train") & labels["label"].isin(["A", "B"]) & ~labels["query_id"].isin(tune_ids)]
ft = lab_tr[["query_id", "query", "label", "pool"]].copy()
ft["gold_pids"] = ft["query_id"].map(gold)
ft["own_pids"] = ft["query_id"].map(own)
ft = ft[ft["gold_pids"].map(lambda g: isinstance(g, list) and len(g) > 0)
        & ~ft["query"].str.lower().isin(val_queries)].reset_index(drop=True)

print("학습 질문:", len(ft))
print(ft.groupby(["pool", "label"]).size().to_string())

In [ ]:
cand_ids, _ = dense_search(base_embedder, base_index, ft["query"].tolist(), k=NEG_DEPTH)
rng = np.random.default_rng(SEED)
neg_pids = []
for cand, own_p in zip(cand_ids, ft["own_pids"]):
    pool = [int(p) for p in cand[NEG_SKIP:] if p not in own_p and p not in val_gold]
    neg_pids.append([int(p) for p in rng.choice(pool, size=min(N_NEG, len(pool)), replace=False)])
ft["neg_pids"] = neg_pids
ft = ft[ft["neg_pids"].map(len) > 0].reset_index(drop=True)

FT_DATA_DIR = DATA_DIR / "finetune"
FT_DATA_DIR.mkdir(parents=True, exist_ok=True)
ft.drop(columns="own_pids").to_parquet(FT_DATA_DIR / "train_pairs.parquet", index=False)
print("학습 질문:", len(ft), "/ 질문당 오답 평균:", ft["neg_pids"].map(len).mean().round(2))

pd.set_option("display.max_colwidth", 160)
for _, r in ft.sample(3, random_state=SEED).iterrows():
    print(f"\nQ: {r['query']}\n  정답: {passages[r['gold_pids'][0]][:160]}\n  오답: {passages[r['neg_pids'][0]][:160]}")

## 4. Dense 파인튜닝 (bge-base)
(질문, 정답, 오답) 묶음과 `MultipleNegativesRankingLoss`로 학습합니다. 배치 안의 다른 질문의 정답도 오답으로 함께 씁니다.

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, losses

embed_examples = [InputExample(texts=[QUERY_PREFIX + q, passages[g[0]], passages[n[0]]])
                  for q, g, n in zip(ft["query"], ft["gold_pids"], ft["neg_pids"])]
embed_loader = DataLoader(embed_examples, shuffle=True, batch_size=EMBED_BATCH)

ft_embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
ft_embedder.fit(
    train_objectives=[(embed_loader, losses.MultipleNegativesRankingLoss(ft_embedder))],
    epochs=EMBED_EPOCHS,
    warmup_steps=int(0.1 * len(embed_loader) * EMBED_EPOCHS),
    optimizer_params={"lr": EMBED_LR},
    use_amp=True,
    show_progress_bar=True,
)
ft_embedder.save(str(EMBED_FT_DIR))
ft_embedder.half()
ft_index = build_index(encode_passages(ft_embedder, passages))   # 모델이 바뀌었으므로 DB 임베딩도 다시 계산
print("saved to", EMBED_FT_DIR)

## 5. Reranker 파인튜닝 (MiniLM cross-encoder)
(질문, 정답)은 1, (질문, 오답)은 0으로 두고 이진 분류로 학습합니다.

In [ ]:
rerank_examples = []
for q, gs, ns in zip(ft["query"], ft["gold_pids"], ft["neg_pids"]):
    rerank_examples += [InputExample(texts=[q, passages[g]], label=1.0) for g in gs]
    rerank_examples += [InputExample(texts=[q, passages[n]], label=0.0) for n in ns]
rerank_loader = DataLoader(rerank_examples, shuffle=True, batch_size=RERANK_BATCH)

ft_reranker = CrossEncoder(RERANK_MODEL, num_labels=1, device="cuda", max_length=512)
ft_reranker.fit(
    train_dataloader=rerank_loader,
    epochs=RERANK_EPOCHS,
    warmup_steps=int(0.1 * len(rerank_loader) * RERANK_EPOCHS),
    optimizer_params={"lr": RERANK_LR},
    use_amp=True,
    show_progress_bar=True,
)
ft_reranker.save(str(RERANK_FT_DIR))
print("학습 쌍:", len(rerank_examples), "/ saved to", RERANK_FT_DIR)

## 6. 파인튜닝 전후 비교
Dense와 Reranker를 각각 원래 모델(base)과 파인튜닝 모델(ft)로 바꿔 네 조합을 비교합니다.
- **검색**: `lab_med_test`의 Recall@10, MRR@10. `lab_med_ext_test`는 정답 passage가 DB에 없는 질문이 대부분이라 검색 점수에서 뺍니다.
- **거절 판단**: 조합마다 임계값을 튜닝 세트에서 다시 정하고, `lab_med_test` 오거절률과 `lab_ood_test` 거절률을 봅니다. Reranker 점수의 범위가 모델마다 달라서 임계값도 따로 정해야 합니다. `lab_med_ext_test` 거절률은 참고로만 봅니다.

In [ ]:
EMBEDDERS = {"base": (base_embedder, base_index), "ft": (ft_embedder, ft_index)}
RERANKERS = {"base": base_reranker, "ft": ft_reranker}
EVAL_SETS = ["lab_med_test", "lab_med_ext_test", "lab_ood_test", "lab_med_tune", "lab_med_ext_tune", "lab_ood_tune"]

bm25_ids = {n: bm25_search(qsets[n]["query"].tolist())[0] for n in EVAL_SETS}
dense_ids, hybrid_ids = {}, {}
for e, (emb, idx) in EMBEDDERS.items():
    for n in EVAL_SETS:
        ids, _ = dense_search(emb, idx, qsets[n]["query"].tolist())
        dense_ids[(e, n)] = [list(map(int, x)) for x in ids]
        hybrid_ids[(e, n)] = rrf_fuse(bm25_ids[n], ids)

rr = {}   # rr[(dense, reranker, set)] = (pid 리스트, 점수 리스트)
for e in EMBEDDERS:
    for r, model in RERANKERS.items():
        for n in EVAL_SETS:
            rr[(e, r, n)] = rerank(model, qsets[n]["query"].tolist(), hybrid_ids[(e, n)])
        print(f"dense={e} reranker={r} done")

In [ ]:
def retrieval_metrics(run, gold, k=K_EVAL):
    recall, mrr = [], []
    for ids, g in zip(run, gold):
        g = set(g)
        top = list(ids)[:k]
        hits = [r for r, pid in enumerate(top) if pid in g]
        recall.append(len(set(top) & g) / len(g))
        mrr.append(1.0 / (hits[0] + 1) if hits else 0.0)
    return {f"Recall@{k}": np.mean(recall), f"MRR@{k}": np.mean(mrr)}

rows = {}
for n in ["lab_med_test"]:
    gold = qsets[n]["gold_pids"].tolist()
    for e in EMBEDDERS:
        rows[(n, f"Dense ({e})")] = retrieval_metrics(dense_ids[(e, n)], gold)
        rows[(n, f"Hybrid ({e})")] = retrieval_metrics(hybrid_ids[(e, n)], gold)
        for r in RERANKERS:
            rows[(n, f"Hybrid ({e}) + Rerank ({r})")] = retrieval_metrics(rr[(e, r, n)][0], gold)
ft_retrieval = pd.DataFrame(rows).T.round(4)
ft_retrieval.to_csv(RESULT_DIR / "retrieval_metrics.csv")
ft_retrieval

In [ ]:
def top1(e, r, n):
    return np.array([s[0] for s in rr[(e, r, n)][1]])

gate_rows = {}
for e in EMBEDDERS:
    for r in RERANKERS:
        med = np.concatenate([top1(e, r, "lab_med_tune"), top1(e, r, "lab_med_ext_tune")])
        ood = top1(e, r, "lab_ood_tune")
        cand = np.unique(np.concatenate([med, ood]))
        bal = [((med >= t).mean() + (ood < t).mean()) / 2 for t in cand]
        thr = float(cand[int(np.argmax(bal))])
        refuse_med = (top1(e, r, "lab_med_test") < thr).mean()
        refuse_ood = (top1(e, r, "lab_ood_test") < thr).mean()
        gate_rows[f"Dense ({e}) + Rerank ({r})"] = {
            "threshold": thr,
            "Refusal_medsite": refuse_med,
            "OODRefusal": refuse_ood,
            "BalancedAcc": (1 - refuse_med + refuse_ood) / 2,
            "Refusal_extsite (참고)": (top1(e, r, "lab_med_ext_test") < thr).mean(),
        }
ft_gate = pd.DataFrame(gate_rows).T.round(4)
ft_gate.to_csv(RESULT_DIR / "gate_metrics.csv")
ft_gate

## 7. 다음 단계
파인튜닝 모델이 더 좋으면 `LabeledBaseline.ipynb`의 `EMBED_MODEL`, `RERANK_MODEL`을 위 저장 경로(`EMBED_FT_DIR`, `RERANK_FT_DIR`)로 바꿔 답변 생성까지 포함한 전체 비교를 다시 실행합니다.